<a href="https://colab.research.google.com/github/2001lida/PythonLession2/blob/hw_10/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B510.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# =========================
# 1. Установка зависимостей
# =========================
!pip install pandas scikit-learn fastapi uvicorn joblib
!npm install -g localtunnel

# =========================
# 2. Обучение модели
# =========================
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# загрузка данных
df = pd.read_csv("realty_data.csv", engine="python", on_bad_lines="skip")

FEATURES = [
    "total_square", "rooms", "floor",
    "city", "district", "area", "object_type",
    "lat", "lon"
]

TARGET = "price"

df = df[FEATURES + [TARGET]].dropna()

X = df[FEATURES]
y = df[TARGET]

num = ["total_square", "rooms", "floor", "lat", "lon"]
cat = ["city", "district", "area", "object_type"]

pre = ColumnTransformer([
    ("num", Pipeline([
        ("imp", SimpleImputer(strategy="median")),
        ("sc", StandardScaler())
    ]), num),

    ("cat", Pipeline([
        ("imp", SimpleImputer(strategy="most_frequent")),
        ("oh", OneHotEncoder(handle_unknown="ignore"))
    ]), cat)
])

model = Pipeline([
    ("pre", pre),
    ("rf", RandomForestRegressor(n_estimators=100))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model.fit(X_train, y_train)

joblib.dump(model, "model.pkl")

print("✅ Модель обучена")



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 22 packages in 1s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋✅ Модель обучена


In [21]:
# =========================
# 3. Создание FastAPI
# =========================
%%writefile main.py
import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

model = joblib.load("model.pkl")

class Input(BaseModel):
    total_square: float
    rooms: int
    floor: int
    city: str
    district: str
    area: str
    object_type: str
    lat: float
    lon: float

def predict(data):
    df = pd.DataFrame([data.dict()])
    return float(model.predict(df)[0])

@app.get("/health")
def health():
    return {"status": "ok"}

@app.get("/predict_get")
def predict_get(
    total_square: float,
    rooms: int,
    floor: int,
    city: str,
    district: str,
    area: str,
    object_type: str,
    lat: float,
    lon: float
):
    data = Input(
        total_square=total_square,
        rooms=rooms,
        floor=floor,
        city=city,
        district=district,
        area=area,
        object_type=object_type,
        lat=lat,
        lon=lon
    )
    return {"prediction": predict(data)}

@app.post("/predict_post")
def predict_post(data: Input):
    return {"prediction": predict(data)}

Overwriting main.py


In [22]:
import subprocess, time

process = subprocess.Popen([
    "uvicorn", "main:app",
    "--host", "0.0.0.0",
    "--port", "8000"
])

time.sleep(5)

print("✅ API запущен")

✅ API запущен


In [ ]:
!lt --port 8000

your url is: https://quick-jars-wish.loca.lt
